# 10 - Long-Running Asynchronous Agents

## Scenario: Background Incident Sweeps

Some agentic workflows take minutes or hours (e.g., scanning 50,000 log lines). If you run this synchronously in a web request, the browser will timeout.

We must decouple the agent using an **Async Job Queue**. The user submits a task, gets a `job_id`, and polls for completion. In this notebook, we simulate a background agent sweep for Northstar Support.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. The Job Queue

We simulate an async queue (like Celery or AWS SQS).

In [2]:
import uuid
import time
import threading

# Simulated Database of Jobs
job_db = {}

def background_agent_task(job_id: str, query: str):
    """This runs in a separate thread/worker process."""
    job_db[job_id] = {"status": "IN_PROGRESS", "result": None}
    print(f"\n⚙️ [Worker] Started job {job_id}: '{query}'")
    
    # Simulate a long-running LLM investigation
    time.sleep(2) # In reality, this might be 15 minutes
    
    job_db[job_id] = {
        "status": "COMPLETED", 
        "result": "Found 3 unauthorized access attempts in the EU region."
    }
    print(f"⚙️ [Worker] Finished job {job_id}")

def submit_agent_job(query: str) -> str:
    """API Endpoint to submit a job."""
    job_id = str(uuid.uuid4())[:8]
    # Fire and forget
    threading.Thread(target=background_agent_task, args=(job_id, query)).start()
    return job_id


## 2. Polling for Completion

In [3]:
print("📩 Submitting complex log sweep request...")
my_job_id = submit_agent_job("Scan all VPC flow logs for the last 30 days for anomalies.")
print(f"✅ Job submitted! ID: {my_job_id}")

# Simulate a client polling the API
for _ in range(4):
    status = job_db.get(my_job_id, {}).get("status", "PENDING")
    print(f"🔄 Polling status... {status}")
    if status == "COMPLETED":
        print(f"🎉 Result: {job_db[my_job_id]['result']}")
        break
    time.sleep(1)


📩 Submitting complex log sweep request...

⚙️ [Worker] Started job bc08da78: 'Scan all VPC flow logs for the last 30 days for anomalies.'
✅ Job submitted! ID: bc08da78
🔄 Polling status... IN_PROGRESS


🔄 Polling status... IN_PROGRESS


⚙️ [Worker] Finished job bc08da78
🔄 Polling status... COMPLETED
🎉 Result: Found 3 unauthorized access attempts in the EU region.


## Checkpoint

**1. Why use Async Job Queues for Agents?**
- A) It makes the LLM hallucinate less.
- B) LLM agents often take a long time to loop through tools and reason. Async queues prevent HTTP timeouts and allow the user to check back later.
- C) It is required by OpenAI's Terms of Service.
- D) It reduces the token cost.
